# Classify all beach cells with best ConvNeXt Tiny

This notebook reads all complete cell matrices from `/Volumes/BACKUPS/Piotr_beach/beach_all`, loads the best saved context-aware ConvNeXt Tiny model, and writes per-date classification CSV files to `Results/all_cells_classification`.

It does not copy any input `.txt` files from the external drive.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "Source").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "Source").exists():
    raise RuntimeError("Could not locate project root containing Source directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Source.inference import (
    CellInferenceDataset,
    discover_complete_cells_from_base_suffix,
    discover_date_directories,
    load_convnext_for_inference,
    load_inference_metadata,
    predict_cells,
)
from Source.training import get_device

In [ ]:
SOURCE_ROOT = Path("/Volumes/BACKUPS/Piotr_beach/beach_all")
MODEL_RESULTS_DIR = PROJECT_ROOT / "Results" / "convnext_tiny"
OUTPUT_ROOT = PROJECT_ROOT / "Results" / "all_cells_classification"

CHECKPOINT_PATH = MODEL_RESULTS_DIR / "convnext_tiny_best.pt"
INFERENCE_METADATA_PATH = MODEL_RESULTS_DIR / "inference_metadata.json"

BATCH_SIZE = 128
NUM_WORKERS = 2
BASE_SUFFIX = "r"

print(f"Project root:       {PROJECT_ROOT}")
print(f"Source root:        {SOURCE_ROOT}")
print(f"Model checkpoint:   {CHECKPOINT_PATH}")
print(f"Inference metadata: {INFERENCE_METADATA_PATH}")
print(f"Output root:        {OUTPUT_ROOT}")

In [ ]:
if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"External source directory not found: {SOURCE_ROOT}")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"ConvNeXt checkpoint not found: {CHECKPOINT_PATH}")

if not INFERENCE_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Inference metadata not found: {INFERENCE_METADATA_PATH}. "
        "Rerun the ConvNeXt training notebook so it saves inference_metadata.json."
    )

metadata = load_inference_metadata(INFERENCE_METADATA_PATH)
idx_to_class_code = {
    int(index): str(class_code)
    for index, class_code in metadata["idx_to_class_code"].items()
}
if "context_radius" in metadata:
    context_radius = int(metadata["context_radius"])
elif "context_radius" in metadata.get("best_config", {}):
    context_radius = int(metadata["best_config"]["context_radius"])
else:
    legacy_context_size = int(metadata.get("context_size", 0))
    context_radius = 0 if legacy_context_size <= 0 else legacy_context_size // 2

print(f"Image size:       {metadata['image_size']}")
print(f"Context radius:   {context_radius}")
print(f"Context window:   {2 * context_radius + 1}x{2 * context_radius + 1}")
print(f"Feature suffixes: {metadata['feature_suffixes']}")
print(f"Class mapping:    {idx_to_class_code}")

In [ ]:
device = get_device()
model = load_convnext_for_inference(CHECKPOINT_PATH, metadata, device=device)
print(f"Device: {device}")

In [ ]:
date_dirs = discover_date_directories(SOURCE_ROOT)
print(f"Date folders found: {len(date_dirs)}")
print("This notebook uses a single-pass workflow: each date folder is scanned once using BASE_SUFFIX, then classified immediately.")

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

written_rows = []
for date_dir in tqdm(date_dirs, desc="Classifying date folders"):
    date = date_dir.name
    cell_ids, incomplete_count = discover_complete_cells_from_base_suffix(
        date_dir,
        feature_suffixes=metadata["feature_suffixes"],
        base_suffix=BASE_SUFFIX,
    )
    if not cell_ids:
        written_rows.append(
            {
                "date": date,
                "classified_cells": 0,
                "incomplete_candidates": incomplete_count,
                "output_path": None,
            }
        )
        continue

    dataset = CellInferenceDataset(
        source_root=SOURCE_ROOT,
        date=date,
        cell_ids=cell_ids,
        image_size=metadata["image_size"],
        mean=metadata["channel_mean"],
        std=metadata["channel_std"],
        feature_suffixes=metadata["feature_suffixes"],
        context_radius=context_radius,
    )
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )

    progress_description = f"Classifying {date} ({len(cell_ids):,} cells)"
    predictions = predict_cells(
        model,
        loader,
        idx_to_class_code,
        device,
        progress_description=progress_description,
    )
    predictions.insert(0, "date", date)
    predictions = predictions[["date", "cell_id", "predicted_class"]]

    output_dir = OUTPUT_ROOT / date
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "classification.csv"
    predictions.to_csv(output_path, index=False)

    written_rows.append(
        {
            "date": date,
            "classified_cells": len(predictions),
            "incomplete_candidates": incomplete_count,
            "output_path": output_path,
        }
    )
    print(
        f"{date}: classified {len(predictions):,} cells, "
        f"skipped {incomplete_count:,} incomplete candidates, "
        f"saved {output_path}"
    )

written_summary = pd.DataFrame(written_rows)
written_summary